In [ ]:
# Program 4: Use word embeddings to improve prompts for Generative AI model. Retrieve similar
# words using word embeddings. Use the similar words to enrich a GenAI prompt. Use the AI model to
# generate responses for the original and enriched prompts. Compare the outputs in terms of detail
# and relevance.

In [ ]:
!pip install transformers -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 93.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 63.7 MB/s eta 0:00:00


In [ ]:
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors

In [ ]:
glove_input_file = "/content/glove.6B.100d.txt"
# Path to GloVe file
word2vec_output_file = "/content/glove.6B.100d.word2vec.txt"
# Output file in Word2Vec format

In [ ]:
glove2word2vec(glove_input_file, word2vec_output_file)

/tmp/ipykernel_4030/694258130.py:1: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(glove_input_file, word2vec_output_file)


(140091, 100)

In [ ]:
# Load the converted Word2Vec model
model = KeyedVectors.load_word2vec_format(word2vec_output_file, binary=False)

# Test the loaded model
print(model.most_similar("king"))

[('prince', 0.7682328820228577), ('queen', 0.7507690787315369), ('son', 0.7020888328552246), ('brother', 0.6985775232315063), ('monarch', 0.6977890729904175), ('throne', 0.6919989585876465), ('kingdom', 0.6811409592628479), ('father', 0.6802029013633728), ('emperor', 0.6712858080863953), ('ii', 0.6676074266433716)]


In [ ]:
# Define the original medical prompt
original_prompt = "Explain the importance of vaccinations in healthcare."

# Define key terms extracted from the original prompt
key_terms = ["vaccinations", "healthcare"]




In [ ]:
similar_terms = []
for term in key_terms:
    if term in model.key_to_index:
        similar_terms.extend({word for word, _ in model.most_similar(term, topn=3)})



In [ ]:
if similar_terms:
    enriched_prompt = f"{original_prompt} Consider aspects like: {', '.join(similar_terms)}."
else:
   enriched_prompt = original_prompt

In [ ]:
# Output the original and enriched prompts
print("Original Prompt:", original_prompt)
print("Enriched Prompt:", enriched_prompt)

Original Prompt: Explain the importance of vaccinations in healthcare.
Enriched Prompt: Explain the importance of vaccinations in healthcare. Consider aspects like: inoculations, vaccination, immunizations, care, services, health.


In [ ]:
import getpass
import os
GOOGLE_API_KEY= os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

Enter your Google AI API key: ··········


In [ ]:
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.7 MB/s eta 0:00:00


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0,
    api_key=GOOGLE_API_KEY,
    max_tokens=256,
    timeout=None,
    max_retries=2,
# other params...
    )
llm.invoke("Hi")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EtEBCs4BAQw51sem94T/enf7oG9lSoMpWhDbajOttv4u0Lwg6485O9jeevyW/GFau71oqsnZl+fBqX7DdxiZh6jH3uXsiyTCtja6CqcJK6sjGE5glgoiq5mXQ2pn7x7VlNQkQ8DtcAuDpQ4dixs8w7cabT9u8JNyd3nhS7WMlz3Ht+J9m309bA7U9N/FIVBcp5DaW+j+/dTjwpb6+mUMtmhPnlje2xsqu/OOclMwvStBTlplw9XYNr9o/cRk6VjxLRDirfkpRAv2BvTEvINJijzaqXA='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e01da-ed9a-76c3-bd2a-9e2944a91df1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 52, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 43}})

In [ ]:
print(llm.invoke(original_prompt).content)

[{'type': 'text', 'text': 'Vaccinations are widely considered one of the greatest achievements', 'extras': {'signature': 'EpoJCpcJAQw51sc0ypuobfnIxpx8+HHUxR27LOeFGDL9a6Bwo93WrOeWKDYPamHi0EaSxD22i4fR7hymPrDSfnPtjkvWNQ4vRyC6DZE+eBHQy2p/KQ+87+Iz2JChpI+G9WP8bo3OdfY3uA/ORcmq5Xq1WGuJwggfU+wO+TAvNa3U5BQH6K8oq636AMt2zXJC3NP0CIJZT4yhhf7XuI8BR6ormk2wLJBpiBaF1nn4tnZq0Sxy5GqF4bzLLyMHHgdMQUeWl0kesf5xlDMSQ22T3Pp5qNBSTyfuuUqwLQ6QdaQx/ziSRkw1ODGVIyGDMAJPTC1UAG1eSIC4RFcAsRlmEG3zrb7Mkf0wX6kPieLxOYM6Yejxx9rhYGl8LnxeshY4GN149719GYqkVdXARtSuU/5YGmE5i1uufhZ/WmHvNG2hzj4hD7rvLmRwAd4PlQUHj/lYMWehCH3dE5YDSPftRcw1JYTv7tm75Uxdff3FQxptvSnPOM0YSMQKO9zaRKLrnJUF1b6NeqwhbA9CwzfrytltwcpbCIv0s8f8HuprE9rnV9HRQ565gpwnbbW6z7a+hTRb/dVScIh/SZq3vyxGUZuUDorvbWdrLwTnj5LmW+Uiw0NTJ13b+lfsE449TSqzyaK2O/JVU0j025EtcqFCoq4pKUkvWbsKP5DvfggV56YY6IDOrFvzOaTcy2JO+fs7Fuh8gyRUrq5oSGcs9bP/LJCJrWWybAr0OWItTfFi+C+yZsJsIp2GbUAX4kk+PGlprdxAdeR8Pe61mHiO7cPssHhkCcXX5EE/Jr/x+XQ9dfAq2rNnqvXAXTULb1FWAF1UT2eBUYgDbCaw5iGuKMrvqXmmYkuIQaIgRoz7b7VO8MPxGI